In [0]:
#dbutils.fs.mkdirs("/FileStore/f1")

In [0]:
races_df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("/Volumes/workspace/f1_proje/f1_data/The_Race.csv")



In [0]:
races_df.select('year')

DataFrame[year: int]

In [0]:
#display(races_df['year'])

In [0]:
races_df['year'].dtype

Column<'year['dtype']'>

In [0]:
#races_df.rdd.getNumPartitions()  # Not supported in serverless, consider using DataFrame API to get partition info

In [0]:
races_df.write.format("delta")\
.mode("overwrite")\
.save("/Volumes/workspace/f1_proje/f1_data/races.delta")


In [0]:
bronze_df=spark.read.format("delta")\
.load("/Volumes/workspace/f1_proje/f1_data/races.delta")
#display(bronze_df)


In [0]:
#display(bronze_df.limit(1))

In [0]:
races_df.write.format("delta")

In [0]:
#display(races_df.limit(2))

In [0]:
races_df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("/Volumes/workspace/f1_proje/f1_data/The_Race.csv")
races_df=races_df.replace("\\N", None)   

In [0]:
drivers_df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("/Volumes/workspace/f1_proje/f1_data/The_Driver.csv")
drivers_df=drivers_df.replace("\\N", None)   

In [0]:
results_df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("/Volumes/workspace/f1_proje/f1_data/The_Result.csv")
results_df = results_df.replace("\\N", None)

In [0]:
constructors_df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("/Volumes/workspace/f1_proje/f1_data/The_Constructor.csv")
constructors_df = constructors_df.replace("\\N", None)    

In [0]:
#display(races_df.limit(1))
#display(drivers_df.limit(1))
#display(results_df.limit(1))
#display(constructors_df.limit(1))


In [0]:
dbutils.fs.mkdirs("/Volumes/workspace/f1_proje/f1_data/bronze")


True

In [0]:
races_df.write.format("delta") \
.mode("overwrite")\
.save("/Volumes/workspace/f1_proje/f1_data/bronze/races")


In [0]:
drivers_df.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/f1_proje/f1_data/bronze/drivers")

In [0]:
results_df.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/f1_proje/f1_data/bronze/results")

In [0]:
constructors_df.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/f1_proje/f1_data/bronze/constructors")

In [0]:
bronze_races_df = spark.read.format("delta") \
    .load("/Volumes/workspace/f1_proje/f1_data/bronze/races")

In [0]:
#A limpeza dos Dados parte da Silver#

In [0]:
#display(bronze_df.limit(1))

In [0]:
#display(bronze_races_df.limit(1))

In [0]:
from pyspark.sql.functions import year, col

silver_races_df = bronze_races_df \
    .withColumn(
        "Ano_corrida",        
         year(col("date"))) \
        .drop("url")

In [0]:
#display(silver_races_df.limit(1))

In [0]:
#silver_races_df.write.format("delta") \
  #  .mode("overwrite") \
   # .save("/Volumes/workspace/f1_proje/f1_data/silver/races")

In [0]:
bronze_drivers_df = spark.read.format("delta") \
    .load("/Volumes/workspace/f1_proje/f1_data/bronze/drivers")

In [0]:
bronze_results_df = spark.read.format("delta") \
    .load("/Volumes/workspace/f1_proje/f1_data/bronze/results")

In [0]:
bronze_constructors_df = spark.read.format("delta") \
    .load("/Volumes/workspace/f1_proje/f1_data/bronze/constructors")

In [0]:
joined_df = bronze_results_df.join(
    bronze_drivers_df,
    bronze_results_df.driverId == bronze_drivers_df.driverId
)

In [0]:
#Pd = races_df.limit(10).toPandas()

In [0]:
#display(Pd)

In [0]:
joined_df

DataFrame[resultId: int, raceId: int, driverId: int, constructorId: int, number: string, grid: int, position: string, positionText: string, positionOrder: int, points: double, laps: int, time: string, milliseconds: string, fastestLap: string, rank: string, fastestLapTime: string, fastestLapSpeed: string, statusId: int, driverId: int, driverRef: string, number: string, code: string, forename: string, surname: string, dob: date, nationality: string, url: string]

In [0]:
bronze_results_df.driverId == bronze_drivers_df.driverId

Column<'==(driverId, driverId)'>

In [0]:
joined_df.explain(True)

In [0]:
#display(joined_df)

In [0]:
joined_df = bronze_results_df.join(
    bronze_drivers_df,
    bronze_results_df.driverId == bronze_drivers_df.driverId
)

In [0]:
#display(joined_df)

In [0]:
clean_joined_df = joined_df.select(
    bronze_results_df.raceId,
    bronze_drivers_df.forename,
    bronze_drivers_df.surname,
    bronze_results_df.position,
    bronze_results_df.points
)

In [0]:
#display(clean_joined_df)

In [0]:
clean_joined_df = joined_df.select(
    bronze_results_df.raceId,
    bronze_drivers_df.forename,
    bronze_drivers_df.surname,
    bronze_results_df.position,
    bronze_results_df.points
)

In [0]:
from pyspark.sql.functions import *

clean_joined_df = clean_joined_df.withColumn(
    "Piloto",
    concat(
        col("forename"),
        lit(" "),
        col("surname")
    )
)

In [0]:
display(clean_joined_df.limit(1))

raceId,forename,surname,position,points,Piloto
18,Lewis,Hamilton,1,10.0,Lewis Hamilton


In [0]:
silver_drivers_df = spark.read.format("delta") \
    .load("/Volumes/workspace/f1_proje/f1_data/silver/drivers")

In [0]:
#display(silver_drivers_df)

In [0]:
spark.read.format("delta")

In [0]:
from pyspark.sql.functions import *

silver_drivers_df = bronze_drivers_df.withColumn(
    "driver_name",
    concat(
        col("forename"),
        lit(" "),
        col("surname") 
    )
).drop("url")

In [0]:
#display(silver_drivers_df.limit(1))

In [0]:
from pyspark.sql.functions import sum

In [0]:
gold_driver_points_df = clean_joined_df.groupBy(
    "Piloto"
).agg(
    sum("points").alias("total_pontos")
)

In [0]:
#display(gold_driver_points_df)

In [0]:
ranking_df=gold_driver_points_df.orderBy(
    col("total_pontos").desc()
)

In [0]:
#display(ranking_df)

In [0]:
ranking_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/f1_proje/f1_data/gold/driver_ranking")

In [0]:
print(spark,version)

<pyspark.sql.connect.session.SparkSession object at 0xff39b62db260> <function version at 0xff3a0c2a8cc0>


In [0]:
from pyspark.sql.functions import col, desc


ranking_df=gold_driver_points_df \
    .filter(col("total_pontos") > 0) \
     .orderBy(desc("total_pontos"))


In [0]:
#display(ranking_df)

In [0]:
bronze_constructors_df=spark.read.format("delta").load("/Volumes/workspace/f1_proje/f1_data/bronze/constructors")

In [0]:
silver_constructors_df=bronze_constructors_df \
    .withColumnRenamed("name", "constructor_name") \
    .drop("url")

In [0]:
#display(silver_constructors_df)

In [0]:
silver_constructors_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/f1_proje/f1_data/silver/constructors")

In [0]:
bronze_results_df=spark.read.format("delta") \
    .load("/Volumes/workspace/f1_proje/f1_data/bronze/results")




In [0]:
#display(bronze_results_df.limit(1))

In [0]:
silver_results_df=bronze_results_df \
    .drop("statusId") \
    .withColumnRenamed("points", "total_pontos") \
    

In [0]:
#display(silver_results_df.limit(1))

In [0]:
silver_results_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/f1_proje/f1_data/silver/results")

In [0]:
#display(silver_results_df.limit(1))

In [0]:
dbutils.fs.ls("/Volumes/workspace/f1_proje/f1_data/silver/results")



#display(silver_results_df.limit(1))


In [0]:
bronze_drivers_df = spark.read.format("delta") \
    .load("/Volumes/workspace/f1_proje/f1_data/bronze/drivers")

In [0]:
silver_drivers_df = bronze_drivers_df \
    .withColumnRenamed("forename", "nome") \
    .withColumnRenamed("surname", "sobrenome") \
    .drop("url")    

In [0]:
#display(silver_drivers_df.limit(1))

In [0]:
silver_drivers_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/f1_proje/f1_data/silver/drivers")

In [0]:
# Bronze  -> dados brutos
# Silver  -> limpeza e transformação
# Gold    -> análises de negócio

In [0]:
silver_drivers_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/f1_proje/f1_data/silver/drivers")

In [0]:
bronze_drivers_df = spark.read.format("delta") \
    .load("/Volumes/workspace/f1_proje/f1_data/bronze/drivers")

silver_drivers_df = bronze_drivers_df \
    .withColumnRenamed("forename", "nome") \
    .withColumnRenamed("surname", "sobrenome") \
    .drop("url")

silver_drivers_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/f1_proje/f1_data/silver/drivers")

In [0]:
#display(silver_drivers_df.limit(1))

In [0]:
bronze_races_df = spark.read.format("delta") \
    .load("/Volumes/workspace/f1_proje/f1_data/bronze/races")

silver_races_df = bronze_races_df \
    .withColumnRenamed("year", "ano_corrida") \
     .drop("url")   

silver_races_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/f1_proje/f1_data/silver/races")

In [0]:
#display(silver_races_df.limit(1))

In [0]:
bronze_results_df = spark.read.format("delta") \
    .load("/Volumes/workspace/f1_proje/f1_data/bronze/results")

silver_results_df = bronze_results_df \
    .withColumnRenamed("points", "total_pontos")

silver_results_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/f1_proje/f1_data/silver/results")

In [0]:
#display(silver_results_df.limit(1))

In [0]:
bronze_constructors_df = spark.read.format("delta") \
    .load("/Volumes/workspace/f1_proje/f1_data/bronze/constructors")

silver_constructors_df = bronze_constructors_df \
    .withColumnRenamed("name", "nome") \
    .withColumnRenamed("nationality", "nacionalidade") \
    .drop("url")
silver_constructors_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/f1_proje/f1_data/silver/constructors")

In [0]:
#display(silver_constructors_df.limit(1 ))

In [0]:
from pyspark.sql.functions import sum, col

silver_results_df = spark.read.format("delta") \
    .load("/Volumes/workspace/f1_proje/f1_data/silver/results")

silver_drivers_df = spark.read.format("delta") \
    .load("/Volumes/workspace/f1_proje/f1_data/silver/drivers")

In [0]:
gold_driver_ranking_df = silver_results_df.groupBy("driverId") \
    .agg(sum("total_pontos").alias("pontuacao_total"))

In [0]:
#display(gold_driver_ranking_df.limit(1))

In [0]:
gold_driver_ranking_df = gold_driver_ranking_df.join(
    silver_drivers_df,
    on="driverId",
    how="inner"
)

In [0]:
gold_driver_ranking_df = gold_driver_ranking_df.orderBy(
    col("pontuacao_total").desc()
)

In [0]:
#display(gold_driver_ranking_df.limit(1))

In [0]:
#display(gold_driver_ranking_df)


In [0]:
gold_driver_ranking_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/f1_proje/f1_data/gold/driver_ranking")

In [0]:
silver_results_df = spark.read.format("delta") \
    .load("/Volumes/workspace/f1_proje/f1_data/silver/results")

In [0]:
silver_constructors_df = spark.read.format("delta") \
    .load("/Volumes/workspace/f1_proje/f1_data/silver/constructors")

In [0]:
silver_constructors_df = bronze_constructors_df.drop("url")

In [0]:
#display(silver_constructors_df.limit(1))

In [0]:
from  pyspark.sql.functions import sum, col

In [0]:
gold_constructor_ranking_df = silver_results_df.groupBy("constructorId") \
    .agg(sum("total_pontos").alias("pontuacao_total"))

In [0]:
gold_constructor_ranking_df = gold_constructor_ranking_df.join(
    silver_constructors_df,
    on="constructorId",
    how="inner"
)

In [0]:
gold_constructor_ranking_df = gold_constructor_ranking_df.orderBy(
    col("pontuacao_total").desc()
)

In [0]:
#display(gold_constructor_ranking_df)


In [0]:
gold_constructor_ranking_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/f1_proje/f1_data/gold/constructor_ranking")

In [0]:
from  pyspark.sql.functions import count

gold_races_year_df = silver_races_df.groupBy("ano_corrida") \
    .agg(count("raceId").alias("total_corridas"))

display(gold_races_year_df)

In [0]:
gold_races_year_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/f1_proje/f1_data/gold/races_by_year")

In [0]:
gold_driver_wins_df = silver_results_df.filter(
    silver_results_df.position == 1
)

In [0]:
gold_driver_wins_df = gold_driver_wins_df.groupBy("driverId") \
    .agg(count("raceId").alias("vitorias"))

In [0]:
gold_driver_wins_df = gold_driver_wins_df.join(
    silver_drivers_df,
    on="driverId",
    how="inner"
)

In [0]:
gold_driver_country_df = silver_drivers_df.groupBy("nationality") \
    .agg(count("driverId").alias("quantidade_pilotos"))

In [0]:
print(gold_driver_country_df)


DataFrame[nationality: string, quantidade_pilotos: bigint]


In [0]:
%sql

Select *
FROM delta.`/Volumes/workspace/f1_proje/f1_data/gold/driver_ranking`
ORDER BY pontuacao_total DESC

In [0]:
%sql
Select nome,
       sobrenome,
       pontuacao_total
FROM delta.`/Volumes/workspace/f1_proje/f1_data/gold/driver_ranking`
LIMIT 10

nome,sobrenome,pontuacao_total
Lewis,Hamilton,4820.5
Sebastian,Vettel,3098.0
Max,Verstappen,2912.5
Fernando,Alonso,2329.0
Kimi,Räikkönen,1873.0
Valtteri,Bottas,1788.0
Nico,Rosberg,1594.5
Sergio,Pérez,1585.0
Michael,Schumacher,1566.0
Charles,Leclerc,1363.0


In [0]:
%sql
SELECT 
*
FROM delta.`/Volumes/workspace/f1_proje/f1_data/gold/constructor_ranking`
ORDER BY pontuacao_total DESC

In [0]:
#O Bronze -> Silver -> Gold

In [0]:
#Foram usados Os comandos Sqls simples